<a href="https://colab.research.google.com/github/Kalpan1104/Phishing-URL-Detector/blob/main/ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.inspection import permutation_importance

import time
import os

In [21]:
print("=" * 60)
print("STEP 1 — Loading Data")
print("=" * 60)

df = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")
df=df.drop(columns=[
    "LineOfCode", "LargestLineLength", "HasTitle", "Title",
    "DomainTitleMatchScore", "URLTitleMatchScore", "HasFavicon",
    "Robots", "IsResponsive", "NoOfURLRedirect", "NoOfSelfRedirect",
    "HasDescription", "NoOfPopup", "NoOfiFrame", "HasExternalFormSubmit",
    "HasSocialNet", "HasSubmitButton", "HasHiddenFields", "HasPasswordField",
    "Bank", "Pay", "Crypto", "HasCopyrightInfo", "NoOfImage", "NoOfCSS",
    "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef",
    "Domain", "TLD", "NoOfLettersInURL", "URLSimilarityIndex","FILENAME", "URL", "Title"
])
print(f"  Shape           : {df.shape}")
print(f"  Memory (MB)     : {df.memory_usage(deep=True).sum() / 1e6:.1f}")
print(f"  Null values     : {df.isnull().sum().sum()}")
print(f"\nLabel distribution:")
vc = df["label"].value_counts()
print(f"  Legitimate (1)  : {vc[1]:,}  ({vc[1]/len(df)*100:.1f}%)")
print(f"  Phishing   (0)  : {vc[0]:,}  ({vc[0]/len(df)*100:.1f}%)")

STEP 1 — Loading Data
  Shape           : (235795, 21)
  Memory (MB)     : 39.6
  Null values     : 0

Label distribution:
  Legitimate (1)  : 134,850  (57.2%)
  Phishing   (0)  : 100,945  (42.8%)


In [22]:
print("\n" + "=" * 60)
print("STEP 2 — Preprocessing")
print("=" * 60)


X = df_model.drop(columns=["label"])
y = df_model["label"]

feature_names = X.columns.tolist()
print(f"  Features used   : {len(feature_names)}")
print(f"  Feature names   : {feature_names[:8]} ...")

# Scale for GBM (RF is invariant to scale, but scaling doesn't hurt)
scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)


STEP 2 — Preprocessing
  Features used   : 50
  Feature names   : ['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength'] ...


In [23]:
print("\n" + "=" * 60)
print("STEP 3 — Train / Test Split")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"  Train size : {X_train.shape[0]:,} rows")
print(f"  Test  size : {X_test.shape[0]:,} rows")
print(f"  Train label balance — Legit: {y_train.sum():,} | Phishing: {(y_train==0).sum():,}")


STEP 3 — Train / Test Split
  Train size : 188,636 rows
  Test  size : 47,159 rows
  Train label balance — Legit: 107,880 | Phishing: 80,756


In [24]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ),
    "Gradient Boosting (XGBoost-equivalent)": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        validation_fraction=0.1,
        n_iter_no_change=20,
        tol=1e-4
    )
}

In [25]:
print("\n" + "=" * 60)
print("STEP 4 — Training & Evaluation")
print("=" * 60)

results     = {}
all_metrics = {}

for name, model in models.items():
    print(f"\n  ── {name} ──")

    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    print(f"  Training time   : {train_time:.1f}s")

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy"  : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred),
        "Recall"    : recall_score(y_test, y_pred),
        "F1-Score"  : f1_score(y_test, y_pred),
        "ROC-AUC"   : roc_auc_score(y_test, y_proba),
        "Avg Precision": average_precision_score(y_test, y_proba),
        "MCC"       : matthews_corrcoef(y_test, y_pred),
        "Train Time": train_time,
    }

    for k, v in metrics.items():
        if k != "Train Time":
            print(f"  {k:<18}: {v:.4f}")

    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Phishing", "Legitimate"]))

    results[name]     = model
    all_metrics[name] = {
        "metrics" : metrics,
        "y_pred"  : y_pred,
        "y_proba" : y_proba,
        "cm"      : confusion_matrix(y_test, y_pred)
    }



STEP 4 — Training & Evaluation

  ── Random Forest ──
  Training time   : 54.6s
  Accuracy          : 1.0000
  Precision         : 1.0000
  Recall            : 1.0000
  F1-Score          : 1.0000
  ROC-AUC           : 1.0000
  Avg Precision     : 1.0000
  MCC               : 1.0000

  Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00     20189
  Legitimate       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159


  ── Gradient Boosting (XGBoost-equivalent) ──
  Training time   : 29.9s
  Accuracy          : 1.0000
  Precision         : 1.0000
  Recall            : 1.0000
  F1-Score          : 1.0000
  ROC-AUC           : 1.0000
  Avg Precision     : 1.0000
  MCC               : 1.0000

  Classification Report:
              precision    recall  f1-score   support

    P

In [26]:
print("\n" + "=" * 60)
print("STEP 5 — 5-Fold Cross-Validation (Random Forest)")
print("=" * 60)

cv_scores = cross_val_score(
    results["Random Forest"], X_scaled, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc", n_jobs=-1
)
print(f"  ROC-AUC  per fold : {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Mean ± Std        : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


STEP 5 — 5-Fold Cross-Validation (Random Forest)
  ROC-AUC  per fold : ['1.0000', '1.0000', '1.0000', '1.0000', '1.0000']
  Mean ± Std        : 1.0000 ± 0.0000


In [27]:
print("\n" + "=" * 60)
print("STEP 6 — Summary Comparison")
print("=" * 60)

metric_keys = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "MCC"]
summary_df  = pd.DataFrame(
    {name: {k: v for k, v in all_metrics[name]["metrics"].items() if k in metric_keys}
     for name in all_metrics}
).T
summary_df = summary_df.round(4)
print(summary_df.to_string())


STEP 6 — Summary Comparison
                                        Accuracy  Precision  Recall  F1-Score  ROC-AUC  MCC
Random Forest                                1.0        1.0     1.0       1.0      1.0  1.0
Gradient Boosting (XGBoost-equivalent)       1.0        1.0     1.0       1.0      1.0  1.0


In [29]:
print("\n" + "=" * 60)
print("STEP 7 — Generating Visualizations")
print("=" * 60)

# Define OUTPUT_DIR and create it if it doesn't exist
OUTPUT_DIR = "output_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

palette = {"Random Forest": "#2563eb",
           "Gradient Boosting (XGBoost-equivalent)": "#dc2626"}
sort   = {"Random Forest": "RF",
           "Gradient Boosting (XGBoost-equivalent)": "GBM"}

# ── Fig 1: Label Distribution ────────────────
fig, ax = plt.subplots(figsize=(6, 4))
colors  = ["#ef4444", "#22c55e"]
bars    = ax.bar(["Phishing (0)", "Legitimate (1)"],
                 [vc[0], vc[1]], color=colors, edgecolor="white", linewidth=1.2)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f"{bar.get_height():,}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Dataset Label Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_label_distribution.png", dpi=150)
plt.close()
print("  Saved fig1_label_distribution.png")

# ── Fig 2: Confusion Matrices ────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, data) in zip(axes, all_metrics.items()):
    cm = data["cm"]
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=ax,
                xticklabels=["Phishing", "Legitimate"],
                yticklabels=["Phishing", "Legitimate"],
                linewidths=0.5)
    ax.set_title(f"Confusion Matrix\n{sort[name]}", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
plt.suptitle("Confusion Matrices — Both Models", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_confusion_matrices.png", dpi=150)
plt.close()
print("  Saved fig2_confusion_matrices.png")

# ── Fig 3: ROC Curves ────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
for name, data in all_metrics.items():
    fpr, tpr, _ = roc_curve(y_test, data["y_proba"])
    auc = data["metrics"]["ROC-AUC"]
    ax.plot(fpr, tpr, label=f"{sort[name]}  (AUC={auc:.4f})",
            color=palette[name], lw=2)
ax.plot([0,1],[0,1], "k--", lw=1, label="Random baseline")
ax.fill_between([0,1],[0,1], alpha=0.05, color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_roc_curves.png", dpi=150)
plt.close()
print("  Saved fig3_roc_curves.png")

# ── Fig 4: Precision-Recall Curves ───────────
fig, ax = plt.subplots(figsize=(7, 6))
for name, data in all_metrics.items():
    prec, rec, _ = precision_recall_curve(y_test, data["y_proba"])
    ap = data["metrics"]["Avg Precision"]
    ax.plot(rec, prec, label=f"{sort[name]}  (AP={ap:.4f})",
            color=palette[name], lw=2)
baseline = y_test.sum() / len(y_test)
ax.axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves", fontsize=14, fontweight="bold")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_pr_curves.png", dpi=150)
plt.close()
print("  Saved fig4_pr_curves.png")

# ── Fig 5: Metrics Comparison Bar Chart ──────
fig, ax = plt.subplots(figsize=(10, 5))
x       = np.arange(len(metric_keys))
width   = 0.35
names   = list(all_metrics.keys())
for i, name in enumerate(names):
    vals = [all_metrics[name]["metrics"][k] for k in metric_keys]
    bars = ax.bar(x + (i - 0.5) * width, vals, width,
                  label=sort[name], color=palette[name], alpha=0.85, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x)
ax.set_xticklabels(metric_keys, fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison", fontsize=14, fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_metrics_comparison.png", dpi=150)
plt.close()
print("  Saved fig5_metrics_comparison.png")

# ── Fig 6: Feature Importances (RF) ──────────
rf_model    = results["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=feature_names)
top20       = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
top20.plot(kind="barh", ax=ax, color="#2563eb", edgecolor="white", alpha=0.85)
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importances — Random Forest", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig6_feature_importances_rf.png", dpi=150)
plt.close()
print("  Saved fig6_feature_importances_rf.png")

# ── Fig 7: Feature Importances (GBM) ─────────
gbm_model    = results["Gradient Boosting (XGBoost-equivalent)"]
importances_g = pd.Series(gbm_model.feature_importances_, index=feature_names)
top20_g       = importances_g.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
top20_g.plot(kind="barh", ax=ax, color="#dc2626", edgecolor="white", alpha=0.85)
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importances — Gradient Boosting", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig7_feature_importances_gbm.png", dpi=150)
plt.close()
print("  Saved fig7_feature_importances_gbm.png")

# ── Fig 8: Feature Correlation Heatmap ───────
top_feats  = importances.nlargest(15).index.tolist()
corr_df    = df_model[top_feats + ["label"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask   = np.zeros_like(corr_df, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.3, ax=ax,
            annot_kws={"size": 7})
ax.set_title("Correlation Heatmap — Top 15 Features + Label",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig8_correlation_heatmap.png", dpi=150)
plt.close()
print("  Saved fig8_correlation_heatmap.png")

# ── Fig 9: Prediction Probability Distribution ─
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, data) in zip(axes, all_metrics.items()):
    proba = data["y_proba"]
    ax.hist(proba[y_test == 1], bins=60, alpha=0.65, label="Legitimate", color="#22c55e")
    ax.hist(proba[y_test == 0], bins=60, alpha=0.65, label="Phishing",   color="#ef4444")
    ax.axvline(0.5, color="black", ls="--", lw=1.2, label="Decision boundary")
    ax.set_xlabel("Predicted Probability (Legitimate)")
    ax.set_ylabel("Count")
    ax.set_title(f"Prediction Distribution — {sort[name]}", fontsize=12, fontweight="bold")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig9_prediction_distributions.png", dpi=150)
plt.close()
print("  Saved fig9_prediction_distributions.png")


STEP 7 — Generating Visualizations
  Saved fig1_label_distribution.png
  Saved fig2_confusion_matrices.png
  Saved fig3_roc_curves.png
  Saved fig4_pr_curves.png
  Saved fig5_metrics_comparison.png
  Saved fig6_feature_importances_rf.png
  Saved fig7_feature_importances_gbm.png
  Saved fig8_correlation_heatmap.png
  Saved fig9_prediction_distributions.png


In [30]:
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(summary_df.to_string())
print(f"\n  5-Fold CV (RF) — ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"\n  All outputs saved to: {OUTPUT_DIR}")
print("=" * 60)
print("Done.")


FINAL SUMMARY
                                        Accuracy  Precision  Recall  F1-Score  ROC-AUC  MCC
Random Forest                                1.0        1.0     1.0       1.0      1.0  1.0
Gradient Boosting (XGBoost-equivalent)       1.0        1.0     1.0       1.0      1.0  1.0

  5-Fold CV (RF) — ROC-AUC: 1.0000 ± 0.0000

  All outputs saved to: output_plots
Done.
